# Traditional EDA

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from helpers.pathlib import get_path_site_checkpoint
from IPython.display import display

from ata_pipeline1.helpers.enums import FieldNew  # , FieldSnowplow
from ata_pipeline1.site import AFRO_LA, DALLAS_FREE_PRESS, OPEN_VALLEJO, THE_19TH

plt.rcdefaults()
plt.style.use("seaborn-v0_8-darkgrid")

In [ ]:
# Nov. 3, 2022 to Jan. 25, 2022 (inclusively) a.k.a 12 weeks
CSV_PREFIX = "221103-230125"

In [ ]:
# Read data from CHECKPOINT 7
df_afla = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, THE_19TH.name))

## Cleaning log

While move along with EDA, list problems indexed by date here. Prepend each listed problem with ✅ once it's solved.

### Feb. 24, 2023
- ✅ `br_viewheight`'s maxima (above 10,000 pixels) and minima (0 pixels) indicate outliers
- ✅ `doc_height`'s maxima (above 100,000 pixels) indicate outliers
- ✅ `dwell_secs`'s minima of 0 is a result of aggregating imperfect data. There needs to be a lower bound for it, depending on page type
- `dwell_secs` also has a couple of outliers. Perhaps would be nice to put an upper bound of 1500 seconds?

## Basic questions

### How many entries do we have on the dataset?

In [ ]:
print(AFRO_LA.name, df_afla.shape[0], "entries")
print(DALLAS_FREE_PRESS.name, df_dfp.shape[0], "entries")
print(OPEN_VALLEJO.name, df_ov.shape[0], "entries")
print(THE_19TH.name, df_19.shape[0], "entries")

### How many columns do we have?

In [ ]:
print(AFRO_LA.name, df_afla.shape[1], "columns")
print(DALLAS_FREE_PRESS.name, df_dfp.shape[1], "columns")
print(OPEN_VALLEJO.name, df_ov.shape[1], "columns")
print(THE_19TH.name, df_19.shape[1], "columns")

### What types of data do we have in our features?

In [ ]:
pd.concat([df_afla, df_dfp, df_ov, df_19]).dtypes.sort_index()

### Do all the columns in the data make sense?

Yes, they do! Refer to `ata_pipeline1/helpers/enums.py` for the column definitions.

## Descriptive statistics & plots

### Quantitative fields

In [ ]:
def describe_and_plot_quant_fields(df):
    df_desc = df.describe().T
    display(df_desc)

    fields = [*df_desc.index]
    num_plots = len(fields)

    df.plot(kind="box", subplots=True, sharex=False, sharey=False, vert=False, layout=(num_plots, 1), figsize=(8, 8))

    plt.tight_layout()
    plt.show()

In [ ]:
describe_and_plot_quant_fields(df_afla)

In [ ]:
describe_and_plot_quant_fields(df_dfp)

In [ ]:
describe_and_plot_quant_fields(df_ov)

In [ ]:
describe_and_plot_quant_fields(df_19)

#### Dwell-time distribution deep dive

In [ ]:
def describe_by_page_types(df, field_to_describe, **kwargs):
    fields_page_types = [f for f in df.columns if "page_is_" in f]

    def describe_by_page_type(df, field_to_describe, field_page_type):
        return (
            df.query(field_page_type)[field_to_describe]
            .describe(**kwargs)
            .rename(f"{field_to_describe}_among_{field_page_type}")
        )

    # Stats table
    display(pd.DataFrame([describe_by_page_type(df, field_to_describe, f) for f in fields_page_types]))

    df_to_plot = pd.concat([df.query(f)[[field_to_describe]].assign(subset=f) for f in fields_page_types])

    # Plot
    plt.figure(figsize=(8, 6))
    sns.violinplot(df_to_plot, y="subset", x=field_to_describe)

    plt.show()

In [ ]:
describe_by_page_types(df_afla, FieldNew.DWELL_SECS)

In [ ]:
describe_by_page_types(df_dfp, FieldNew.DWELL_SECS)

In [ ]:
describe_by_page_types(df_ov, FieldNew.DWELL_SECS)

In [ ]:
describe_by_page_types(df_19, FieldNew.DWELL_SECS)

### Qualitative fields

#### Boolean fields

In [ ]:
fields_boolean = [
    FieldNew.DVCE_IS_MOBILE,
    FieldNew.FORM_SUBMIT_IS_NEWSLETTER,
    FieldNew.PAGE_IS_ABOUT_US,
    FieldNew.PAGE_IS_ARTICLE,
    FieldNew.PAGE_IS_AUTHOR_PROFILE,
    FieldNew.PAGE_IS_DONATION,
    FieldNew.PAGE_IS_HOME,
    FieldNew.PAGE_IS_NEWSLETTER,
    FieldNew.PAGE_IS_SECTION,
    FieldNew.LEAD_TO_NEWSLETTER_CONVERSION,
]

In [ ]:
def describe_and_plot_bool_fields(df):
    def add_perc_columns(row):
        row = row.copy()
        values = row.index
        row["total"] = row.sum()
        for v in values:
            row[f"perc_{v}"] = (row[v] / row["total"]) * 100

        return row

    # Show value-count table
    df_summary = (
        df[fields_boolean]
        .apply(lambda x: x.value_counts(dropna=False))
        .T.replace([np.nan], [0])
        .astype(int)
        .apply(add_perc_columns, axis=1)
    )
    display(df_summary)

    # Plot value counts
    df_summary[[f for f in df_summary.columns if "perc_" in str(f)]].plot(
        kind="barh", figsize=(6, 6), stacked=True
    ).legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
    plt.show()

In [ ]:
describe_and_plot_bool_fields(df_afla)

In [ ]:
describe_and_plot_bool_fields(df_dfp)

In [ ]:
describe_and_plot_bool_fields(df_ov)

In [ ]:
describe_and_plot_bool_fields(df_19)

### Time-series profling

### Target variable analysis

### Target variable analysis